<a href="https://colab.research.google.com/github/kilianodonell-cmd/Durban/blob/main/Field_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ── Install packages ─────────────────────────────────────────
!pip install folium rasterio geopandas numpy Pillow -q


In [2]:

# ── Mount Google Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 1 — Setup
# Field Map — Housing Suitability
# Mzinyati Stream Catchment, eThekwini Municipality
#
# Reads outputs from Durban_MCA.ipynb.
# Exports a single HTML file for use on a tablet in the field.
# Requires:
#   - outputs/rasters/suitability_*.tif   (from Cell 13)
#   - outputs/rasters/constraint_mask.tif (from Cell 11)
#   - outputs/outputs/at_risk_buildings.gpkg (from Cell 14)
#   - outputs/field_map_config.json       (from Cell 2)
# ============================================================
import os, json, numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import folium
from folium import LayerControl
from PIL import Image
import base64, io

CONFIG_PATH = '/content/drive/MyDrive/Durban/outputs/field_map_config.json'

with open(CONFIG_PATH) as f:
    config = json.load(f)

OUTPUT_ROOT    = config['OUTPUT_ROOT']
TARGET_CRS     = config['TARGET_CRS']
SCENARIOS      = config['SCENARIOS']
SUIT_COLORS    = config['SUIT_COLORS']
SUIT_LABELS    = config['SUIT_LABELS']
RASTERS_DIR    = os.path.join(OUTPUT_ROOT, 'rasters')
BUILDINGS_PATH = os.path.join(OUTPUT_ROOT, 'outputs', 'at_risk_buildings.gpkg')

print("Config loaded.")
print(f"  Output root: {OUTPUT_ROOT}")
print(f"  Scenarios:   {list(SCENARIOS.keys())}")


Config loaded.
  Output root: /content/drive/MyDrive/Durban/outputs
  Scenarios:   ['hazard_focused', 'balanced', 'infrastructure_focused']


In [4]:
# ============================================================
# CELL 2 — Build and Export Field Map
# Builds an interactive HTML map for tablet field use.
#
# Map contains:
#   - Satellite + street label basemap
#   - Suitability layers (one per scenario, toggleable)
#   - Constraint areas (always visible, not toggleable)
#   - Building footprints (toggleable)
#   - AOI boundary (always visible)
#
# Output: outputs/field_map.html
# ============================================================

print("Building field map...")

# ============================================================
# HELPER — Convert raster to PNG overlay for folium
# ============================================================

def raster_to_png_overlay(data, colors_hex, bounds_wgs84):
    """
    Convert a scored raster to a transparent PNG for folium overlay.
    data       : 2D numpy array of scores 1-5, NaN = transparent
    colors_hex : list of 5 hex colours matching scores 1-5
    bounds_wgs84: [[south, west], [north, east]]
    Returns folium.raster_layers.ImageOverlay
    """
    h, w   = data.shape
    rgba   = np.zeros((h, w, 4), dtype=np.uint8)

    for i, hex_color in enumerate(colors_hex):
        score = i + 1
        mask  = (data >= score - 0.5) & (data < score + 0.5)
        r     = int(hex_color[1:3], 16)
        g     = int(hex_color[3:5], 16)
        b     = int(hex_color[5:7], 16)
        rgba[mask] = [r, g, b, 180]  # 180/255 = ~70% opacity

    img    = Image.fromarray(rgba, 'RGBA')
    buffer = io.BytesIO()
    img.save(buffer, format='PNG')
    buffer.seek(0)
    encoded = base64.b64encode(buffer.read()).decode('utf-8')
    png_url = f"data:image/png;base64,{encoded}"

    return folium.raster_layers.ImageOverlay(
        image   = png_url,
        bounds  = bounds_wgs84,
        opacity = 1.0,
        name    = None,
    )

# ============================================================
# LOAD AOI — for map centre and boundary
# ============================================================

print("  Loading AOI...")

catchments = gpd.read_file(config['CATCHMENTS_PATH'])
aoi        = catchments[catchments[config['AOI_FIELD']] == config['AOI_VALUE']]
aoi_wgs84  = aoi.to_crs('EPSG:4326')
bounds     = aoi_wgs84.total_bounds  # minx, miny, maxx, maxy
centre     = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

# Bounds for folium — [[south, west], [north, east]]
folium_bounds = [[bounds[1], bounds[0]], [bounds[3], bounds[2]]]

print(f"  AOI centre: {centre[0]:.4f}, {centre[1]:.4f}")

# ============================================================
# LOAD RASTERS — reproject to WGS84 for folium
# ============================================================

print("  Loading suitability rasters...")

scenario_rasters = {}

for scenario_name in SCENARIOS:
    path = os.path.join(RASTERS_DIR, f"suitability_{scenario_name}.tif")
    if not os.path.exists(path):
        print(f"  ✗ {scenario_name} not found — run Cell 13 first")
        continue

    with rasterio.open(path) as src:
        # Reproject to WGS84
        transform_wgs, width_wgs, height_wgs = calculate_default_transform(
            src.crs, 'EPSG:4326', src.width, src.height, *src.bounds
        )
        data_wgs = np.zeros((height_wgs, width_wgs), dtype='float32')
        reproject(
            source        = rasterio.band(src, 1),
            destination   = data_wgs,
            src_transform = src.transform,
            src_crs       = src.crs,
            dst_transform = transform_wgs,
            dst_crs       = 'EPSG:4326',
            resampling    = Resampling.nearest
        )

    data_wgs[data_wgs <= 0] = np.nan
    scenario_rasters[scenario_name] = data_wgs
    print(f"  ✓ {scenario_name}")

# ============================================================
# LOAD CONSTRAINT MASK
# ============================================================

print("  Loading constraint mask...")

with rasterio.open(os.path.join(RASTERS_DIR, 'constraint_mask.tif')) as src:
    transform_wgs, width_wgs, height_wgs = calculate_default_transform(
        src.crs, 'EPSG:4326', src.width, src.height, *src.bounds
    )
    constraint_wgs = np.zeros((height_wgs, width_wgs), dtype='float32')
    reproject(
        source        = rasterio.band(src, 1),
        destination   = constraint_wgs,
        src_transform = src.transform,
        src_crs       = src.crs,
        dst_transform = transform_wgs,
        dst_crs       = 'EPSG:4326',
        resampling    = Resampling.nearest
    )

# ============================================================
# LOAD BUILDINGS
# ============================================================

print("  Loading building footprints...")

buildings = gpd.read_file(BUILDINGS_PATH).to_crs('EPSG:4326')
print(f"  ✓ {len(buildings):,} buildings loaded")

# ============================================================
# BUILD FOLIUM MAP
# ============================================================

print("  Building map...")

# ── Base map ──────────────────────────────────────────────────
m = folium.Map(
    location        = centre,
    zoom_start      = 14,
    tiles           = None,
    control_scale   = True,
)

# ── Satellite basemap ─────────────────────────────────────────
folium.TileLayer(
    tiles     = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr      = 'Esri World Imagery',
    name      = 'Satellite',
    overlay   = False,
    control   = False,
).add_to(m)

# ── Street labels overlay ─────────────────────────────────────
folium.TileLayer(
    tiles     = 'https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}',
    attr      = 'Esri Labels',
    name      = 'Labels',
    overlay   = True,
    control   = False,
).add_to(m)

# ── AOI boundary — always visible ────────────────────────────
folium.GeoJson(
    aoi_wgs84.__geo_interface__,
    name  = 'AOI Boundary',
    style_function = lambda x: {
        'color'      : 'black',
        'weight'     : 2.5,
        'fillOpacity': 0,
    },
    control = False,
).add_to(m)

# ── Constraint mask — always visible, not toggleable ─────────
constraint_rgba           = np.zeros((*constraint_wgs.shape, 4), dtype=np.uint8)
excluded                  = constraint_wgs == 0
constraint_rgba[excluded] = [220, 38, 38, 180]  # red, semi-transparent

constraint_img    = Image.fromarray(constraint_rgba, 'RGBA')
constraint_buffer = io.BytesIO()
constraint_img.save(constraint_buffer, format='PNG')
constraint_buffer.seek(0)
constraint_encoded = base64.b64encode(constraint_buffer.read()).decode('utf-8')

folium.raster_layers.ImageOverlay(
    image   = f"data:image/png;base64,{constraint_encoded}",
    bounds  = folium_bounds,
    opacity = 1.0,
    name    = 'Constraint Areas',
    show    = True,
).add_to(m)

folium.Marker(
    location = [bounds[1] + 0.001, bounds[0] + 0.001],
    icon     = folium.DivIcon(html='''
        <div style="background:rgba(220,38,38,0.7); color:white;
                    padding:3px 6px; border-radius:3px; font-size:11px;
                    white-space:nowrap;">
            ⛔ Constraint area — no development
        </div>
    '''),
    popup = None,
).add_to(m)

# ── Suitability layers — one per scenario, toggleable ─────────
for scenario_name, description in SCENARIOS.items():
    if scenario_name not in scenario_rasters:
        continue

    data    = scenario_rasters[scenario_name]
    overlay = raster_to_png_overlay(data, SUIT_COLORS, folium_bounds)

    fg = folium.FeatureGroup(
        name = f"Suitability: {description}",
        show = scenario_name == 'hazard_focused'  # default on
    )
    overlay.name = f"Suitability: {description}"
    overlay.add_to(fg)
    fg.add_to(m)

# ── Building footprints — toggleable ─────────────────────────
buildings_fg = folium.FeatureGroup(name='Existing Buildings', show=True)

# Colour buildings by suitability score of first scenario
score_col = f"score_{list(SCENARIOS.keys())[0]}"
if score_col in buildings.columns:
    for _, row in buildings.iterrows():
        score = row.get(score_col, -1)
        if score in [1,2,3,4,5]:
            color = SUIT_COLORS[int(score)-1]
        elif row.get('in_constraint', False):
            color = '#dc2626'
        else:
            color = '#6b7280'

        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function = lambda x, c=color: {
                'fillColor'  : c,
                'color'      : '#333333',
                'weight'     : 0.5,
                'fillOpacity': 0.8,
            }
        ).add_to(buildings_fg)
else:
    folium.GeoJson(
        buildings.__geo_interface__,
        name  = 'Existing Buildings',
        style_function = lambda x: {
            'fillColor'  : '#6b7280',
            'color'      : '#333333',
            'weight'     : 0.5,
            'fillOpacity': 0.7,
        }
    ).add_to(buildings_fg)

buildings_fg.add_to(m)

# ── Legend ────────────────────────────────────────────────────
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 12px 16px; border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.3); font-family: Arial;
            font-size: 12px; max-width: 220px;">
  <b style="font-size:13px;">Housing Suitability</b><br>
  <i style="font-size:10px; color:#666;">Mzinyati Stream Catchment</i>
  <hr style="margin:6px 0;">
'''
for i, (color, label) in enumerate(zip(SUIT_COLORS, SUIT_LABELS)):
    legend_html += f'''
  <div style="margin:3px 0;">
    <span style="display:inline-block; width:16px; height:16px;
                 background:{color}; border:1px solid #ccc;
                 vertical-align:middle; margin-right:6px;"></span>
    {label}
  </div>'''

legend_html += '''
  <div style="margin:3px 0;">
    <span style="display:inline-block; width:16px; height:16px;
                 background:#dc2626; border:1px solid #ccc;
                 vertical-align:middle; margin-right:6px;"></span>
    Constraint area
  </div>
  <hr style="margin:6px 0;">
  <i style="font-size:10px; color:#666;">
    Toggle layers using the ⊞ button (top right)
  </i>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# ── Layer control ─────────────────────────────────────────────
LayerControl(
    position  = 'topright',
    collapsed = False,
).add_to(m)

# ── Fit map to AOI ────────────────────────────────────────────
m.fit_bounds(folium_bounds)

# ============================================================
# SAVE HTML
# ============================================================

out_path = os.path.join(OUTPUT_ROOT, 'field_map.html')
m.save(out_path)

print(f"\n  ✓ Field map saved → field_map.html")
print(f"  Open on tablet: {out_path}")
print(f"\n  Map contains:")
print(f"    - Satellite basemap with street labels")
print(f"    - {len(scenario_rasters)} suitability scenario layers (toggleable)")
print(f"    - Constraint areas (always visible)")
print(f"    - {len(buildings):,} building footprints (toggleable)")
print(f"    - AOI boundary")

Building field map...
  Loading AOI...
  AOI centre: -29.6664, 30.8913
  Loading suitability rasters...
  ✓ hazard_focused
  ✓ balanced
  ✓ infrastructure_focused
  Loading constraint mask...
  Loading building footprints...
  ✓ 22,196 buildings loaded
  Building map...


/tmp/ipykernel_1662/981310383.py:184: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  constraint_img    = Image.fromarray(constraint_rgba, 'RGBA')
/tmp/ipykernel_1662/981310383.py:40: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img    = Image.fromarray(rgba, 'RGBA')



  ✓ Field map saved → field_map.html
  Open on tablet: /content/drive/MyDrive/Durban/outputs/field_map.html

  Map contains:
    - Satellite basemap with street labels
    - 3 suitability scenario layers (toggleable)
    - Constraint areas (always visible)
    - 22,196 building footprints (toggleable)
    - AOI boundary
